In [15]:
import pandas as pd

df = pd.read_csv("arxiv_data.csv")
df.head(10)

,titles,summaries,terms
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"['cs.CV', 'cs.LG']"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"['cs.CV', 'cs.AI', 'cs.LG']"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","['cs.CV', 'cs.AI']"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,['cs.CV']
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","['cs.CV', 'cs.LG']"
5,EdgeFlow: Achieving Practical Interactive Segm...,High-quality training data play a key role in ...,"['cs.CV', 'cs.HC']"
6,Efficient Hybrid Transformer: Learning Global-...,Semantic segmentation of fine-resolution urban...,['cs.CV']
7,Towards to Robust and Generalized Medical Imag...,"To mitigate the radiologist's workload, comput...","['cs.CV', 'cs.AI']"
8,Semi-supervised Meta-learning with Disentangle...,Generalising deep models to new data from new ...,['cs.CV']
9,Semi-supervised Contrastive Learning for Label...,The success of deep learning methods in medica...,['cs.CV']


In [16]:
print(df.shape)
print(type(df["terms"].iloc[0]))

(51774, 3)
<class 'str'>


In [17]:
import ast
from collections import Counter
from sklearn.preprocessing import MultiLabelBinarizer

# 在原始 terms 上统计文档频次；每篇文章中同一标签只计一次。
parsed_labels = df["terms"].map(
    lambda value: ast.literal_eval(value) if isinstance(value, str) else value
)
label_counts = Counter(
    label for labels in parsed_labels for label in set(labels)
)
top_9_labels = [label for label, _ in label_counts.most_common(9)]
top_9_set = set(top_9_labels)
original_size = len(df)

# 先直接修改 df：删除非前九类标签，再删除不含前九类的文章。
# terms 现在存储实际列表；保留的文章仍可含多个标签，不设置 Other 类。
df["terms"] = parsed_labels.map(
    lambda labels: list(dict.fromkeys(
        label for label in labels if label in top_9_set
    ))
)
df = df.loc[df["terms"].str.len().gt(0)].copy().reset_index(drop=True)

# 显式固定九个输出位置，顺序与 top_9_labels 一致。
mlb = MultiLabelBinarizer(classes=top_9_labels)
label_matrix = mlb.fit_transform(df["terms"])
df["labels"] = label_matrix.tolist()

print("保留的九大类别：", top_9_labels)
print("原始文章数量：", original_size)
print("保留文章数量：", len(df))
print("删除文章数量：", original_size - len(df))
print("Multi-hot 矩阵形状：", label_matrix.shape)
print("向量位置对应的标签顺序：", mlb.classes_.tolist())
display(df[["terms", "labels"]].head(10))


保留的九大类别： ['cs.CV', 'cs.LG', 'stat.ML', 'cs.AI', 'eess.IV', 'cs.RO', 'cs.CL', 'cs.NE', 'cs.CR']
原始文章数量： 51774
保留文章数量： 51774
删除文章数量： 0
Multi-hot 矩阵形状： (51774, 9)
向量位置对应的标签顺序： ['cs.CV', 'cs.LG', 'stat.ML', 'cs.AI', 'eess.IV', 'cs.RO', 'cs.CL', 'cs.NE', 'cs.CR']


,terms,labels
0,"[cs.CV, cs.LG]","[1, 1, 0, 0, 0, 0, 0, 0, 0]"
1,"[cs.CV, cs.AI, cs.LG]","[1, 1, 0, 1, 0, 0, 0, 0, 0]"
2,"[cs.CV, cs.AI]","[1, 0, 0, 1, 0, 0, 0, 0, 0]"
3,[cs.CV],"[1, 0, 0, 0, 0, 0, 0, 0, 0]"
4,"[cs.CV, cs.LG]","[1, 1, 0, 0, 0, 0, 0, 0, 0]"
5,[cs.CV],"[1, 0, 0, 0, 0, 0, 0, 0, 0]"
6,[cs.CV],"[1, 0, 0, 0, 0, 0, 0, 0, 0]"
7,"[cs.CV, cs.AI]","[1, 0, 0, 1, 0, 0, 0, 0, 0]"
8,[cs.CV],"[1, 0, 0, 0, 0, 0, 0, 0, 0]"
9,[cs.CV],"[1, 0, 0, 0, 0, 0, 0, 0, 0]"


In [18]:
df.head(10)

,titles,summaries,terms,labels
0,Survey on Semantic Stereo Matching / Semantic ...,Stereo matching is one of the widely used tech...,"[cs.CV, cs.LG]","[1, 1, 0, 0, 0, 0, 0, 0, 0]"
1,FUTURE-AI: Guiding Principles and Consensus Re...,The recent advancements in artificial intellig...,"[cs.CV, cs.AI, cs.LG]","[1, 1, 0, 1, 0, 0, 0, 0, 0]"
2,Enforcing Mutual Consistency of Hard Regions f...,"In this paper, we proposed a novel mutual cons...","[cs.CV, cs.AI]","[1, 0, 0, 1, 0, 0, 0, 0, 0]"
3,Parameter Decoupling Strategy for Semi-supervi...,Consistency training has proven to be an advan...,[cs.CV],"[1, 0, 0, 0, 0, 0, 0, 0, 0]"
4,Background-Foreground Segmentation for Interio...,"To ensure safety in automated driving, the cor...","[cs.CV, cs.LG]","[1, 1, 0, 0, 0, 0, 0, 0, 0]"
5,EdgeFlow: Achieving Practical Interactive Segm...,High-quality training data play a key role in ...,[cs.CV],"[1, 0, 0, 0, 0, 0, 0, 0, 0]"
6,Efficient Hybrid Transformer: Learning Global-...,Semantic segmentation of fine-resolution urban...,[cs.CV],"[1, 0, 0, 0, 0, 0, 0, 0, 0]"
7,Towards to Robust and Generalized Medical Imag...,"To mitigate the radiologist's workload, comput...","[cs.CV, cs.AI]","[1, 0, 0, 1, 0, 0, 0, 0, 0]"
8,Semi-supervised Meta-learning with Disentangle...,Generalising deep models to new data from new ...,[cs.CV],"[1, 0, 0, 0, 0, 0, 0, 0, 0]"
9,Semi-supervised Contrastive Learning for Label...,The success of deep learning methods in medica...,[cs.CV],"[1, 0, 0, 0, 0, 0, 0, 0, 0]"


In [19]:
print(df.shape)

(51774, 4)


In [20]:
print(label_matrix.shape)

(51774, 9)


In [21]:
# 基于整理后的 df 定义训练数据，labels 保留九维 multi-hot 编码。
training_df = df[["summaries", "labels"]].copy()
training_df.insert(
    0, "document_id", [f"arxiv_{i}" for i in range(len(df))]
)
training_df = training_df.rename(columns={"summaries": "training_text"})
training_df = training_df.dropna(subset=["training_text", "labels"])
training_df["training_text"] = (
    training_df["training_text"].astype(str)
    .str.replace(r"\s+", " ", regex=True).str.strip()
)
training_df = training_df.loc[
    training_df["training_text"].ne("")
].reset_index(drop=True)

assert training_df["document_id"].is_unique
assert training_df["labels"].map(
    lambda values: len(values) == len(mlb.classes_)
    and all(value in (0, 1) for value in values)
    and sum(values) > 0
).all()

print("Training dataframe shape:", training_df.shape)
print("Label order:", mlb.classes_.tolist())
display(training_df.head())


Training dataframe shape: (51774, 3)
Label order: ['cs.CV', 'cs.LG', 'stat.ML', 'cs.AI', 'eess.IV', 'cs.RO', 'cs.CL', 'cs.NE', 'cs.CR']


,document_id,training_text,labels
0,arxiv_0,Stereo matching is one of the widely used tech...,"[1, 1, 0, 0, 0, 0, 0, 0, 0]"
1,arxiv_1,The recent advancements in artificial intellig...,"[1, 1, 0, 1, 0, 0, 0, 0, 0]"
2,arxiv_2,"In this paper, we proposed a novel mutual cons...","[1, 0, 0, 1, 0, 0, 0, 0, 0]"
3,arxiv_3,Consistency training has proven to be an advan...,"[1, 0, 0, 0, 0, 0, 0, 0, 0]"
4,arxiv_4,"To ensure safety in automated driving, the cor...","[1, 1, 0, 0, 0, 0, 0, 0, 0]"


In [22]:
import os
import gc
import time
import numpy as np
import pandas as pd
import torch

from scipy.special import expit

from iterstrat.ml_stratifiers import (
    MultilabelStratifiedKFold
)

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    precision_score,
    recall_score,
)

from datasets import Dataset, disable_progress_bars
from pathlib import Path
from transformers.trainer_callback import PrinterCallback, ProgressCallback
from transformers.utils import logging as transformers_logging

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)


MODELS = [
    "google-bert/bert-base-uncased",
    "allenai/scibert_scivocab_uncased",
    "nlpaueb/legal-bert-base-uncased",
    "ahmedrachid/FinancialBERT",
    "Lianglab/PharmBERT-uncased",
    "recobo/agriculture-bert-uncased",
    "recobo/chemical-bert-uncased",
]

# 使用现有 training_df 的编码，避免沿用过滤前 label_matrix 的行顺序。
training_label_matrix = np.asarray(training_df["labels"].tolist(), dtype=np.float32)
NUM_LABELS = len(mlb.classes_)
assert training_label_matrix.shape == (len(training_df), NUM_LABELS)
assert np.isin(training_label_matrix, [0, 1]).all()
assert (training_label_matrix.sum(axis=1) > 0).all()
disable_progress_bars()
transformers_logging.disable_progress_bar()
N_SPLITS = 5
TRAINING_SEED = 42

NUM_EPOCHS = 3
TRAIN_BATCH_SIZE = 16
EVAL_BATCH_SIZE = 16
LEARNING_RATE = 2e-5
WEIGHT_DECAY = 0.01

# 必须与之前EnvDataset一致
MAX_LENGTH = 128

OUTPUT_ROOT = (
    str((Path.cwd() / "arXiv" if (Path.cwd() / "arXiv" / "arxiv_data.csv").exists() else Path.cwd()) / "results" / "arxiv_9class_5fold")
)

FOLD_RESULT_CSV = os.path.join(
    OUTPUT_ROOT,
    "arxiv_9class_5fold_results.csv",
)

os.makedirs(
    OUTPUT_ROOT,
    exist_ok=True,
)
# 保存输出位置与类别名称的对应关系，不改变数据。
import json
Path(OUTPUT_ROOT, "label_names.json").write_text(
    json.dumps(mlb.classes_.tolist(), ensure_ascii=False, indent=2),
    encoding="utf-8",
)


105

In [23]:
def calculate_multilabel_metrics(
    logits,
    true_labels,
    threshold=0.5,
):
    probabilities = expit(logits)

    predicted_labels = (
        probabilities >= threshold
    ).astype(int)

    true_labels = (
        true_labels.astype(int)
    )

    return {
        # Multi-label accuracy_score
        # means exact/subset accuracy
        "accuracy": accuracy_score(
            true_labels,
            predicted_labels,
        ),

        "macro_f1": f1_score(
            true_labels,
            predicted_labels,
            average="macro",
            zero_division=0,
        ),

        "micro_f1": f1_score(
            true_labels,
            predicted_labels,
            average="micro",
            zero_division=0,
        ),

        "macro_precision":
            precision_score(
                true_labels,
                predicted_labels,
                average="macro",
                zero_division=0,
            ),

        "macro_recall":
            recall_score(
                true_labels,
                predicted_labels,
                average="macro",
                zero_division=0,
            ),
    }

In [24]:
def evaluate_model_5fold_multilabel(
    model_name,
    full_df,
    label_matrix,
    n_splits=5,
):
    print("\nEvaluating:", model_name)

    splitter = (
        MultilabelStratifiedKFold(
            n_splits=n_splits,
            shuffle=True,
            random_state=TRAINING_SEED,
        )
    )

    tokenizer = (
        AutoTokenizer
        .from_pretrained(
            model_name,
            use_fast=True,
        )
    )

    # Tokenize the complete dataset once
    complete_dataset = Dataset.from_dict({
        "text": (
            full_df["training_text"]
            .tolist()
        ),
        "labels": (
            label_matrix
            .astype(np.float32)
            .tolist()
        ),
    })

    def tokenize_batch(batch):
        return tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_LENGTH,
        )

    tokenized_dataset = (
        complete_dataset.map(
            tokenize_batch,
            batched=True,
            remove_columns=["text"],
            desc=f"Tokenizing {model_name}",
        )
    )

    tokenized_dataset.set_format(
        type="torch"
    )

    data_collator = (
        DataCollatorWithPadding(
            tokenizer=tokenizer,
            padding="longest",
        )
    )

    fold_results = []

    for fold, (
        train_indices,
        test_indices,
    ) in enumerate(
        splitter.split(
            np.zeros(
                (len(full_df), 1)
            ),
            label_matrix,
        ),
        start=1,
    ):
        print(
            f"\n========== "
            f"Fold {fold}/{n_splits} "
            f"=========="
        )

        set_seed(TRAINING_SEED)

        train_dataset = (
            tokenized_dataset.select(
                train_indices.tolist()
            )
        )

        test_dataset = (
            tokenized_dataset.select(
                test_indices.tolist()
            )
        )

        model = (
            AutoModelForSequenceClassification
            .from_pretrained(
                model_name,
                num_labels=NUM_LABELS,
                problem_type=(
                    "multi_label_classification"
                ),
            )
        )

        safe_model_name = (
            model_name
            .replace("/", "__")
        )

        output_dir = os.path.join(
            OUTPUT_ROOT,
            (
                f"{safe_model_name}"
                f"_fold_{fold}"
            ),
        )

        training_args = (
            TrainingArguments(
                output_dir=output_dir,

                eval_strategy="no",
                save_strategy="no",

                logging_strategy="no",

                per_device_train_batch_size=(
                    TRAIN_BATCH_SIZE
                ),

                per_device_eval_batch_size=(
                    EVAL_BATCH_SIZE
                ),

                num_train_epochs=(
                    NUM_EPOCHS
                ),

                learning_rate=(
                    LEARNING_RATE
                ),

                weight_decay=(
                    WEIGHT_DECAY
                ),

                seed=TRAINING_SEED,
                data_seed=TRAINING_SEED,

                report_to="none",

                # Prevent notebook callback error
                disable_tqdm=True,

                fp16=False,
                bf16=False,

                use_cpu=(
                    not torch.cuda.is_available()
                ),
            )
        )

        trainer = Trainer(
            model=model,
            args=training_args,
            train_dataset=train_dataset,
            data_collator=data_collator,
        )

        trainer.remove_callback(PrinterCallback)
        trainer.remove_callback(ProgressCallback)
        start_time = time.time()

        train_result = trainer.train()

        predictions = trainer.predict(
            test_dataset
        )

        logits = predictions.predictions

        if isinstance(logits, tuple):
            logits = logits[0]

        metrics = (
            calculate_multilabel_metrics(
                logits=logits,
                true_labels=(
                    predictions.label_ids
                ),
                threshold=0.5,
            )
        )

        elapsed_hours = (
            time.time() - start_time
        ) / 3600

        fold_row = {
            "dataset": "arxiv_9class",
            "model": model_name,
            "fold": fold,

            "accuracy":
                metrics["accuracy"],

            "macro_f1":
                metrics["macro_f1"],

            "micro_f1":
                metrics["micro_f1"],

            "macro_precision":
                metrics[
                    "macro_precision"
                ],

            "macro_recall":
                metrics[
                    "macro_recall"
                ],

            "training_loss":
                train_result.training_loss,

            "training_hours":
                elapsed_hours,

            "training_seed":
                TRAINING_SEED,

            "epochs":
                NUM_EPOCHS,

            "max_length":
                MAX_LENGTH,

            "train_batch_size":
                TRAIN_BATCH_SIZE,

            "learning_rate":
                LEARNING_RATE,

            "weight_decay":
                WEIGHT_DECAY,

            "threshold":
                0.5,

            "train_size":
                len(train_dataset),

            "test_size":
                len(test_dataset),
        }

        fold_results.append(
            fold_row
        )

        print(
            f"Fold {fold} Accuracy: "
            f"{metrics['accuracy']:.4f}"
        )

        print(
            f"Fold {fold} Macro F1: "
            f"{metrics['macro_f1']:.4f}"
        )

        print(
            f"Fold {fold} Micro F1: "
            f"{metrics['micro_f1']:.4f}"
        )

        print(
            f"Fold {fold} Hours: "
            f"{elapsed_hours:.2f}"
        )

        # Save immediately after every fold
        new_fold_df = pd.DataFrame(
            fold_results
        )
        previous = pd.read_csv(FOLD_RESULT_CSV) if os.path.exists(FOLD_RESULT_CSV) else pd.DataFrame()
        if not previous.empty:
            previous = previous.loc[previous["model"] != model_name]
        saved_results = pd.concat([previous, new_fold_df], ignore_index=True)
        temporary_csv = Path(FOLD_RESULT_CSV).with_suffix(".tmp")
        saved_results.to_csv(temporary_csv, index=False)
        temporary_csv.replace(FOLD_RESULT_CSV)


        del model
        del trainer
        del train_dataset
        del test_dataset

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    fold_df = pd.DataFrame(
        fold_results
    )

    return fold_df

In [25]:
if os.path.exists(
    FOLD_RESULT_CSV
):
    all_fold_results = pd.read_csv(
        FOLD_RESULT_CSV
    )
else:
    all_fold_results = (
        pd.DataFrame()
    )


for model_name in MODELS:
    completed_folds = set()

    if not all_fold_results.empty:
        model_previous_results = (
            all_fold_results[
                all_fold_results["model"]
                == model_name
            ]
        )

        completed_folds = set(
            model_previous_results[
                "fold"
            ].astype(int)
        )

    if len(completed_folds) == N_SPLITS:
        print(
            "Skipping completed model:",
            model_name,
        )
        continue

    model_fold_df = (
        evaluate_model_5fold_multilabel(
            model_name=model_name,
            full_df=training_df,
            label_matrix=training_label_matrix,
            n_splits=N_SPLITS,
        )
    )

    # Remove older rows for this model
    if not all_fold_results.empty:
        all_fold_results = (
            all_fold_results[
                all_fold_results["model"]
                != model_name
            ]
        )

    all_fold_results = pd.concat(
        [
            all_fold_results,
            model_fold_df,
        ],
        ignore_index=True,
    )

    all_fold_results.to_csv(
        FOLD_RESULT_CSV,
        index=False,
    )

    print(
        "Saved:",
        FOLD_RESULT_CSV,
    )


Evaluating: google-bert/bert-base-uncased



========== Fold 1/5 ==========


BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 1 Accuracy: 0.5371
Fold 1 Macro F1: 0.5087
Fold 1 Micro F1: 0.8041
Fold 1 Hours: 0.12

========== Fold 2/5 ==========


BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Fold 2 Accuracy: 0.5478
Fold 2 Macro F1: 0.5226
Fold 2 Micro F1: 0.8065
Fold 2 Hours: 0.12

========== Fold 3/5 ==========


... (62 outputs left out)

BertForSequenceClassification LOAD REPORT from: recobo/chemical-bert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider t

Fold 4 Accuracy: 0.5382
Fold 4 Macro F1: 0.4692
Fold 4 Micro F1: 0.8000
Fold 4 Hours: 0.12

========== Fold 5/5 ==========


BertForSequenceClassification LOAD REPORT from: recobo/chemical-bert-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.bias                     | MISSING    | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 
bert.pooler.dense.weight                   | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider t

Fold 5 Accuracy: 0.5214
Fold 5 Macro F1: 0.4918
Fold 5 Micro F1: 0.7928
Fold 5 Hours: 0.12
Saved: /home/amyliu/Desktop/GR/arXiv/results/arxiv_9class_5fold/arxiv_9class_5fold_results.csv


In [26]:
import pandas as pd

# Read saved 5-fold results
all_fold_results = pd.read_csv(
    FOLD_RESULT_CSV
)

# Average the five folds for each model
real_model_ranking = (
    all_fold_results
    .groupby(
        "model",
        as_index=False,
    )
    .agg(
        completed_folds=(
            "fold",
            "nunique",
        ),

        macro_f1_mean=(
            "macro_f1",
            "mean",
        ),
        macro_f1_std=(
            "macro_f1",
            "std",
        ),

        micro_f1_mean=(
            "micro_f1",
            "mean",
        ),
        micro_f1_std=(
            "micro_f1",
            "std",
        ),

        accuracy_mean=(
            "accuracy",
            "mean",
        ),
        accuracy_std=(
            "accuracy",
            "std",
        ),

        macro_precision_mean=(
            "macro_precision",
            "mean",
        ),

        macro_recall_mean=(
            "macro_recall",
            "mean",
        ),

        training_hours_total=(
            "training_hours",
            "sum",
        ),
    )
)

# Keep only models with all five folds
real_model_ranking = (
    real_model_ranking[
        real_model_ranking[
            "completed_folds"
        ] == N_SPLITS
    ]
    .copy()
)

# Rank by mean Macro-F1
real_model_ranking[
    "real_rank"
] = (
    real_model_ranking[
        "macro_f1_mean"
    ]
    .rank(
        ascending=False,
        method="min",
    )
    .astype(int)
)

# Sort by real rank
real_model_ranking = (
    real_model_ranking
    .sort_values(
        "real_rank"
    )
    .reset_index(drop=True)
)

# Put rank first
real_model_ranking = (
    real_model_ranking[
        [
            "real_rank",
            "model",
            "macro_f1_mean",
            "macro_f1_std",
            "micro_f1_mean",
            "micro_f1_std",
            "accuracy_mean",
            "accuracy_std",
            "macro_precision_mean",
            "macro_recall_mean",
            "completed_folds",
            "training_hours_total",
        ]
    ]
)

display(real_model_ranking)

,real_rank,model,macro_f1_mean,macro_f1_std,micro_f1_mean,micro_f1_std,accuracy_mean,accuracy_std,macro_precision_mean,macro_recall_mean,completed_folds,training_hours_total
0,1,allenai/scibert_scivocab_uncased,0.563098,0.006205,0.817458,0.001526,0.564908,0.004975,0.688338,0.525239,5,0.602256
1,2,recobo/agriculture-bert-uncased,0.559670,0.012400,0.816042,0.001612,0.561954,0.003450,0.685482,0.523105,5,0.584280
2,3,Lianglab/PharmBERT-uncased,0.524315,0.008297,0.805744,0.002736,0.544543,0.007541,0.627572,0.490297,5,0.581838
3,4,google-bert/bert-base-uncased,0.518951,0.008129,0.805834,0.002184,0.541784,0.006039,0.632016,0.490942,5,0.584263
4,5,ahmedrachid/FinancialBERT,0.506256,0.008733,0.798102,0.001236,0.537925,0.005645,0.660153,0.466053,5,0.582250
5,6,nlpaueb/legal-bert-base-uncased,0.502278,0.004159,0.798311,0.001153,0.536296,0.006767,0.635362,0.465578,5,0.583976
6,7,recobo/chemical-bert-uncased,0.482794,0.008898,0.795519,0.002748,0.526946,0.007449,0.617343,0.450279,5,0.583798


In [27]:
REAL_RANKING_CSV = os.path.join(
    OUTPUT_ROOT,
    "arxiv_9class_real_model_ranking.csv",
)

real_model_ranking.to_csv(
    REAL_RANKING_CSV,
    index=False,
)

print("Saved to:", REAL_RANKING_CSV)

Saved to: /home/amyliu/Desktop/GR/arXiv/results/arxiv_9class_5fold/arxiv_9class_real_model_ranking.csv
